# Neural Networks / MLPs: multiclass digit classification

This notebook implements a Multi-Layer Perceptron (MLP) for digit classification using the
Optical Digits dataset (10 classes, 64 features).

Following the `models.md` plan:
1. **scikit-learn baseline** — `MLPClassifier` for quick reference results.
2. **NumPy from scratch** — XOR proof-of-concept, then a full one-hidden-layer MLP
   with forward pass, backpropagation, mini-batch SGD, and manual metrics.

The same dataset and evaluation pattern used in the Softmax Regression notebook is followed here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling is essential for neural networks — it stabilizes gradients and speeds up convergence.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = MLPClassifier(
    hidden_layer_sizes=(128,),   # one hidden layer with 128 neurons
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Log loss: {log_loss(y_test, y_proba):.4f}')
print('\nClassification report:\n', classification_report(y_test, y_pred))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=digits.target_names).plot(cmap='Blues')
plt.title('MLP (sklearn): confusion matrix')
plt.show()

In [ ]:
# Plot learning curve and sample predictions
fig = plt.figure(figsize=(14, 5))

# Loss curve
ax1 = fig.add_subplot(121)
ax1.plot(model.loss_curve_, color='indigo', lw=2)
ax1.set_title('Training Loss Curve')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')

# Sample predictions
import matplotlib.gridspec as gridspec
inner = gridspec.GridSpecFromSubplotSpec(3, 4, subplot_spec=ax1.get_subplotspec()._subplot.get_gridspec()[0, 1], wspace=0.1, hspace=0.5)

for i in range(12):
    ax = plt.Subplot(fig, inner[i])
    fig.add_subplot(ax)
    img = X_test[i].reshape(8, 8)
    ax.imshow(img, cmap='gray')
    color = 'green' if y_test[i] == y_pred[i] else 'red'
    ax.set_title(f'P:{y_pred[i]} T:{y_test[i]}', color=color, fontsize=10)
    ax.axis('off')
    
plt.suptitle('Neural Network Diagnostics', fontsize=16)
plt.show()


## MLP: model, training, backpropagation, and evaluation

### Notation

- $X \in \mathbb{R}^{n \times d}$: feature matrix with $n$ examples and $d=64$ pixel features.
- $\mathbf{x}_i$: features for digit image $i$; $y_i \in \{0, \ldots, 9\}$: its true class.
- $K=10$: number of output classes.
- $h$: number of neurons in the hidden layer (e.g., 128).
- $\mathbf{W}^{(1)} \in \mathbb{R}^{d \times h}$: weights from input to hidden layer; $\mathbf{b}^{(1)} \in \mathbb{R}^{h}$: hidden biases.
- $\mathbf{W}^{(2)} \in \mathbb{R}^{h \times K}$: weights from hidden to output layer; $\mathbf{b}^{(2)} \in \mathbb{R}^{K}$: output biases.

### Forward pass

**Hidden layer** — linear transform followed by ReLU activation:

$$\mathbf{z}^{(1)}_i = \mathbf{x}_i \mathbf{W}^{(1)} + \mathbf{b}^{(1)}$$

$$\mathbf{a}^{(1)}_i = \mathrm{ReLU}(\mathbf{z}^{(1)}_i) = \max(0, \mathbf{z}^{(1)}_i)$$

ReLU is used because it avoids the vanishing-gradient problem of sigmoid/tanh: its gradient is either $0$ or $1$, so gradients flow through active neurons unchanged.

**Output layer** — linear transform followed by softmax:

$$\mathbf{z}^{(2)}_i = \mathbf{a}^{(1)}_i \mathbf{W}^{(2)} + \mathbf{b}^{(2)}$$

$$p_{ik} = \frac{e^{z^{(2)}_{ik}}}{\sum_{j=1}^{K} e^{z^{(2)}_{ij}}}$$

$$\hat{y}_i = \operatorname*{argmax}_{k} p_{ik}$$

### Training objective: multiclass cross-entropy

$$J = -\frac{1}{n}\sum_{i=1}^{n} \log(p_{i,y_i})$$

Same loss as softmax regression. **Minimize** it: $0$ is best.

### Backpropagation

Backpropagation computes $\frac{\partial J}{\partial \mathbf{W}^{(l)}}$ and $\frac{\partial J}{\partial \mathbf{b}^{(l)}}$ by applying the chain rule layer by layer, from output back to input.

**Output layer gradients:**

$$\delta^{(2)} = \mathbf{P} - \mathbf{Y}_{\text{one-hot}}$$

$$\frac{\partial J}{\partial \mathbf{W}^{(2)}} = \frac{1}{n} \mathbf{A}^{(1)T} \delta^{(2)}, \qquad \frac{\partial J}{\partial \mathbf{b}^{(2)}} = \frac{1}{n} \sum_i \delta^{(2)}_i$$

**Hidden layer gradients:**

$$\delta^{(1)} = \delta^{(2)} {\mathbf{W}^{(2)}}^T \odot \mathbb{1}[\mathbf{z}^{(1)} > 0]$$

$$\frac{\partial J}{\partial \mathbf{W}^{(1)}} = \frac{1}{n} \mathbf{X}^T \delta^{(1)}, \qquad \frac{\partial J}{\partial \mathbf{b}^{(1)}} = \frac{1}{n} \sum_i \delta^{(1)}_i$$

where $\odot$ is element-wise multiplication and $\mathbb{1}[\mathbf{z}^{(1)} > 0]$ is the ReLU derivative (1 where pre-activation is positive, 0 otherwise).

### Weight initialization: He init

$$W^{(l)}_{ij} \sim \mathcal{N}\left(0,\; \frac{2}{\mathrm{fan\_in}}\right)$$

He initialization scales weights by $\sqrt{2/\mathrm{fan\_in}}$, where $\mathrm{fan\_in}$ is the number of input neurons to that layer. This keeps the variance of activations stable across layers when using ReLU, preventing both vanishing and exploding gradients at the start of training. Biases are initialized to zero.

### Mini-batch stochastic gradient descent (SGD)

Instead of computing the gradient over all $n$ training examples (batch GD) or a single example (stochastic GD), mini-batch SGD uses a subset of size $B$ (e.g., 64):

$$\mathbf{W}^{(l)} \leftarrow \mathbf{W}^{(l)} - \eta \cdot \frac{\partial J_{\mathcal{B}}}{\partial \mathbf{W}^{(l)}}$$

$$\mathbf{b}^{(l)} \leftarrow \mathbf{b}^{(l)} - \eta \cdot \frac{\partial J_{\mathcal{B}}}{\partial \mathbf{b}^{(l)}}$$

where $\eta$ is the learning rate and $J_{\mathcal{B}}$ is the loss computed over mini-batch $\mathcal{B}$.

**Why mini-batch?** It balances noise (which helps escape local minima) with computational efficiency (matrix operations on batches are faster than single-sample updates). Each pass through the entire training set is called an **epoch**; the data is shuffled at the start of each epoch.

### Why ReLU over sigmoid for hidden layers

| Property | Sigmoid | ReLU |
| --- | --- | --- |
| Range | $(0, 1)$ | $[0, \infty)$ |
| Output | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $\max(0, z)$ |
| Gradient | $\sigma(z)(1-\sigma(z))$, max $0.25$ | $0$ or $1$ |
| Vanishing gradient | Yes — saturates for large $|z|$ | No — gradient is $1$ for $z > 0$ |
| Computation | Expensive ($\exp$) | Cheap ($\max$) |

Sigmoid is still used in the **output layer** for binary classification (logistic regression). For multi-class, we use softmax.

### Feature scaling: standardization

$$x_{ij}^{\mathrm{scaled}} = \frac{x_{ij}-\mu_j}{\sigma_j}$$

$$\mu_j = \frac{1}{n}\sum_{i=1}^{n}x_{ij}, \qquad \sigma_j = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_{ij}-\mu_j)^2}$$

$\mu_j$ and $\sigma_j$ are calculated from training data only, then the same values transform test data. Scaling is **essential** for neural networks — without it, neurons receiving large-magnitude inputs saturate or produce exploding gradients, making training unstable.

### Finite-difference gradient check

To verify backpropagation is correct, compare each analytic gradient against a numerical approximation:

$$\frac{\partial J}{\partial w} \approx \frac{J(w + \epsilon) - J(w - \epsilon)}{2\epsilon}$$

The relative error between the analytic and numerical gradient should be below $10^{-5}$:

$$\text{relative error} = \frac{|g_{\text{analytic}} - g_{\text{numerical}}|}{|g_{\text{analytic}}| + |g_{\text{numerical}}| + \epsilon_{\text{small}}}$$

### Test-set evaluation

$$\mathrm{Accuracy} = \frac{1}{m}\sum_{i=1}^{m}\mathbb{1}(\hat{y}_i=y_i)$$

For each class $k$ in a multiclass problem:

$$\mathrm{Precision}_k = \frac{TP_k}{TP_k+FP_k}, \qquad \mathrm{Recall}_k = \frac{TP_k}{TP_k+FN_k}$$

$$F_{1,k} = 2\cdot\frac{\mathrm{Precision}_k \cdot \mathrm{Recall}_k}{\mathrm{Precision}_k + \mathrm{Recall}_k}$$

where $TP_k$, $FP_k$, $FN_k$ are true positives, false positives, and false negatives for class $k$ in a one-vs-rest sense.

**Maximize** Accuracy, Precision, Recall, and $F_1$: each ranges from $0$ (worst) to $1$ (best). The confusion matrix shows actual classes by row and predicted classes by column.

## Neural Network from scratch (NumPy only)

Following the `models.md` plan: **XOR first** (to verify forward/backward pass),
then the **Optical Digits** dataset with a one-hidden-layer MLP.

Implements:
- Forward pass with ReLU hidden activation and softmax output
- Backpropagation with chain rule
- Mini-batch SGD
- Manual evaluation metrics

In [ ]:
import numpy as np

# ── XOR dataset (non-linearly separable — needs a hidden layer) ─────
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])   # 2 classes

# ── Helpers ──────────────────────────────────────────────────────────
def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def softmax(Z):
    exp_Z = np.exp(Z - Z.max(axis=1, keepdims=True))
    return exp_Z / exp_Z.sum(axis=1, keepdims=True)

def one_hot(y, K):
    oh = np.zeros((len(y), K))
    oh[np.arange(len(y)), y] = 1.0
    return oh

def cross_entropy_loss(P, y):
    return -np.mean(np.log(P[np.arange(len(y)), y] + 1e-9))

# ── XOR MLP: 2 → 8 → 2 ─────────────────────────────────────────────
np.random.seed(42)
d, h, K = 2, 8, 2
W1 = np.random.randn(d, h) * 0.5
b1 = np.zeros(h)
W2 = np.random.randn(h, K) * 0.5
b2 = np.zeros(K)
Y_oh = one_hot(y_xor, K)

lr = 0.5
for epoch in range(2000):
    # Forward
    Z1 = X_xor @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    P = softmax(Z2)

    # Backward
    n = len(X_xor)
    dZ2 = (P - Y_oh) / n
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(axis=0)
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_deriv(Z1)
    dW1 = X_xor.T @ dZ1
    db1 = dZ1.sum(axis=0)

    # Update
    W1 -= lr * dW1;  b1 -= lr * db1
    W2 -= lr * dW2;  b2 -= lr * db2

    if epoch % 500 == 0:
        loss = cross_entropy_loss(P, y_xor)
        preds = np.argmax(P, axis=1)
        acc = np.mean(preds == y_xor)
        print(f'Epoch {epoch:4d}: loss = {loss:.4f}, acc = {acc:.0%}')

# Final predictions
Z1 = X_xor @ W1 + b1; A1 = relu(Z1); Z2 = A1 @ W2 + b2; P = softmax(Z2)
preds = np.argmax(P, axis=1)
print(f'\nFinal XOR predictions: {preds}  (expected: [0 1 1 0])')
print(f'Probabilities:\n{np.round(P, 3)}')

In [ ]:
import numpy as np
from sklearn.datasets import load_digits   # only for loading the data

# ── Load data ────────────────────────────────────────────────────────
digits = load_digits()
X, y = digits.data, digits.target   # X: (1797, 64), y: 0–9

# ── Stratified train/test split (pure NumPy) ─────────────────────────
def stratified_split_numpy(X, y, test_size=0.2, seed=42):
    np.random.seed(seed)
    train_idx, test_idx = [], []
    for label_value in np.unique(y):
        class_indices = np.where(y == label_value)[0]
        shuffled = np.random.permutation(class_indices)
        n_test = int(len(shuffled) * test_size)
        test_idx.extend(shuffled[:n_test])
        train_idx.extend(shuffled[n_test:])
    train_idx = np.random.permutation(train_idx)
    test_idx  = np.random.permutation(test_idx)
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = stratified_split_numpy(X, y, test_size=0.2, seed=42)

# ── Manual feature scaling (fit on train only) ───────────────────────
def fit_scaler(X):
    mean = X.mean(axis=0)
    std  = X.std(axis=0)
    std[std == 0] = 1.0
    return mean, std

def apply_scaler(X, mean, std):
    return (X - mean) / std

mean, std = fit_scaler(X_train)
X_train_scaled = apply_scaler(X_train, mean, std)
X_test_scaled  = apply_scaler(X_test, mean, std)

# ── Helpers ──────────────────────────────────────────────────────────
def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def softmax(Z):
    exp_Z = np.exp(Z - Z.max(axis=1, keepdims=True))
    return exp_Z / exp_Z.sum(axis=1, keepdims=True)

def one_hot(y, K):
    oh = np.zeros((len(y), K))
    oh[np.arange(len(y)), y] = 1.0
    return oh

def cross_entropy_loss(P, y):
    return -np.mean(np.log(P[np.arange(len(y)), y] + 1e-9))

# ── He initialization ────────────────────────────────────────────────
# He init: scale weights by sqrt(2/fan_in). Designed for ReLU to keep
# variance stable across layers and prevent vanishing/exploding gradients.
def he_init(fan_in, fan_out):
    return np.random.randn(fan_in, fan_out) * np.sqrt(2.0 / fan_in)

# ── MLP: 64 → 128 → 10 (one hidden layer) ───────────────────────────
np.random.seed(42)
d = X_train_scaled.shape[1]   # 64
h = 128                        # hidden neurons
K = len(np.unique(y_train))   # 10

W1 = he_init(d, h);   b1 = np.zeros(h)
W2 = he_init(h, K);   b2 = np.zeros(K)

# ── Training with mini-batch SGD ─────────────────────────────────────
lr = 0.1
n_epochs = 50
batch_size = 64
n_samples = X_train_scaled.shape[0]

for epoch in range(n_epochs):
    # Shuffle training data each epoch
    perm = np.random.permutation(n_samples)
    X_shuf = X_train_scaled[perm]
    y_shuf = y_train[perm]

    epoch_loss = 0.0
    n_batches = 0

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        X_b = X_shuf[start:end]
        y_b = y_shuf[start:end]
        mb = len(y_b)
        Y_oh = one_hot(y_b, K)

        # ── Forward pass ─────────────────────────────────────────
        Z1 = X_b @ W1 + b1          # (mb, h)
        A1 = relu(Z1)                # (mb, h)
        Z2 = A1 @ W2 + b2            # (mb, K)
        P  = softmax(Z2)             # (mb, K)

        # ── Backward pass ────────────────────────────────────────
        # Output layer
        dZ2 = (P - Y_oh) / mb        # (mb, K)
        dW2 = A1.T @ dZ2             # (h, K)
        db2 = dZ2.sum(axis=0)        # (K,)

        # Hidden layer
        dA1 = dZ2 @ W2.T             # (mb, h)
        dZ1 = dA1 * relu_deriv(Z1)   # (mb, h)
        dW1 = X_b.T @ dZ1            # (d, h)
        db1 = dZ1.sum(axis=0)        # (h,)

        # ── Update weights ───────────────────────────────────────
        W1 -= lr * dW1;  b1 -= lr * db1
        W2 -= lr * dW2;  b2 -= lr * db2

        epoch_loss += cross_entropy_loss(P, y_b) * mb
        n_batches += 1

    epoch_loss /= n_samples

    if epoch % 10 == 0 or epoch == n_epochs - 1:
        # Compute training accuracy
        Z1_full = X_train_scaled @ W1 + b1
        A1_full = relu(Z1_full)
        Z2_full = A1_full @ W2 + b2
        P_full  = softmax(Z2_full)
        train_acc = np.mean(np.argmax(P_full, axis=1) == y_train)
        print(f'Epoch {epoch:3d}: loss = {epoch_loss:.4f}, train_acc = {train_acc:.4f}')

# ── Predict on test set ──────────────────────────────────────────────
Z1_t = X_test_scaled @ W1 + b1
A1_t = relu(Z1_t)
Z2_t = A1_t @ W2 + b2
P_t  = softmax(Z2_t)
y_pred = np.argmax(P_t, axis=1)

# ── Manual metrics ───────────────────────────────────────────────────
accuracy = np.mean(y_pred == y_test)
test_loss = cross_entropy_loss(P_t, y_test)

print(f'\nTest accuracy:      {accuracy:.4f}')
print(f'Test cross-entropy: {test_loss:.4f}')

# Per-class precision, recall, F1
print(f'\n{"Class":>5}  {"Precision":>9}  {"Recall":>6}  {"F1":>6}  {"Support":>7}')
print('-' * 44)
for k in range(K):
    tp = np.sum((y_pred == k) & (y_test == k))
    fp = np.sum((y_pred == k) & (y_test != k))
    fn = np.sum((y_pred != k) & (y_test == k))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    support = int(np.sum(y_test == k))
    print(f'{k:5d}  {prec:9.4f}  {rec:6.4f}  {f1:6.4f}  {support:7d}')

# Confusion matrix
cm = np.zeros((K, K), dtype=int)
for true, pred in zip(y_test, y_pred):
    cm[true, pred] += 1
print(f'\nConfusion matrix (rows = actual, cols = predicted):\n{cm}')

## Finite-difference gradient check

As recommended in `models.md`, we verify backpropagation by comparing analytic gradients
against numerical (finite-difference) gradients on a small subset. If the relative error
is below $10^{-5}$, backprop is implemented correctly.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits

# Use a tiny subset for the gradient check
digits = load_digits()
X_gc = digits.data[:8].copy()
y_gc = digits.target[:8].copy()
# Scale
X_gc = (X_gc - X_gc.mean(axis=0)) / (X_gc.std(axis=0) + 1e-8)

def relu(z):  return np.maximum(0, z)

def softmax(Z):
    e = np.exp(Z - Z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def one_hot(y, K):
    oh = np.zeros((len(y), K)); oh[np.arange(len(y)), y] = 1.0; return oh

def compute_loss(X, y, W1, b1, W2, b2):
    Z1 = X @ W1 + b1; A1 = relu(Z1)
    Z2 = A1 @ W2 + b2; P = softmax(Z2)
    return -np.mean(np.log(P[np.arange(len(y)), y] + 1e-9))

np.random.seed(0)
d, h, K = 64, 16, 10
W1 = np.random.randn(d, h) * 0.1; b1 = np.zeros(h)
W2 = np.random.randn(h, K) * 0.1; b2 = np.zeros(K)

# Analytic gradients via backprop
n = len(X_gc)
Z1 = X_gc @ W1 + b1; A1 = relu(Z1)
Z2 = A1 @ W2 + b2; P = softmax(Z2)
Y_oh = one_hot(y_gc, K)
dZ2 = (P - Y_oh) / n
dW2_analytic = A1.T @ dZ2
dA1 = dZ2 @ W2.T
dZ1 = dA1 * (Z1 > 0).astype(float)
dW1_analytic = X_gc.T @ dZ1

# Numerical gradients via finite differences
eps = 1e-5
for name, W, dW_analytic in [('W1', W1, dW1_analytic), ('W2', W2, dW2_analytic)]:
    dW_numerical = np.zeros_like(W)
    # Check a random subset of 20 entries to keep it fast
    indices = list(zip(*[np.random.randint(0, s, 20) for s in W.shape]))
    max_rel_err = 0.0
    for (i, j) in indices:
        old = W[i, j]
        W[i, j] = old + eps
        loss_plus = compute_loss(X_gc, y_gc, W1, b1, W2, b2)
        W[i, j] = old - eps
        loss_minus = compute_loss(X_gc, y_gc, W1, b1, W2, b2)
        W[i, j] = old
        numerical = (loss_plus - loss_minus) / (2 * eps)
        analytic = dW_analytic[i, j]
        rel_err = abs(numerical - analytic) / (abs(numerical) + abs(analytic) + 1e-8)
        max_rel_err = max(max_rel_err, rel_err)
    print(f'{name}: max relative error = {max_rel_err:.2e}  {"✓ PASS" if max_rel_err < 1e-5 else "✗ FAIL"}')